# Fase CRISP-DM: Análisis Exploratorio de Datos (EDA)
**Plataforma**: AgroData Intelligence Platform (AgroStatsApp)  
**Propósito**: Diagnosticar la estructura distributiva de las variables agropecuarias, analizar correlaciones, detectar anomalías multivariadas e inspeccionar dinámicas temporales y estacionales de precios y variables biofísicas.

In [ ]:
# 1. Configuración de Entorno Resiliente (Google Colab / VS Code / Jupyter Local)
import os
import sys
import subprocess
from pathlib import Path

def setup_environment():
    # A. Detección y preparación automática para Google Colab
    if 'google.colab' in sys.modules or Path('/content').exists():
        print('[INFO] Entorno detectado: Google Colab.')
        repo_dir = Path('/content/Statsfirm')
        if not repo_dir.exists():
            print('[INFO] Clonando repositorio oficial Statsfirm en Colab...')
            subprocess.run(['git', 'clone', 'https://github.com/adansanchezc1-spec/Statsfirm.git', '/content/Statsfirm'], check=True)
        else:
            print('[INFO] Actualizando repositorio en Colab...')
            subprocess.run(['git', '-C', '/content/Statsfirm', 'pull'], check=False)
        
        app_dir = repo_dir / 'AgroStats AndTech' / 'AgroStatsApp'
        if app_dir.exists():
            os.chdir(str(app_dir))
            src_dir = app_dir / 'src'
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            print(f'[OK] Directorio de trabajo establecido en: {app_dir}')
            print(f'[OK] Carpeta src agregada a sys.path: {src_dir}')
            return

    # B. Detección dinámica en Entorno Local (Windows / Linux / WSL / VS Code)
    candidates = [
        Path.cwd() / 'src',
        Path.cwd() / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path.cwd().parent / 'src',
        Path.cwd().parent.parent / 'src',
        Path.cwd().parent.parent / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path(r'c:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\AgroStatsApp\src'),
        Path('/mnt/c/Users/ADAN/OneDrive/Documentos/Statsfirm/AgroStats AndTech/AgroStatsApp/src'),
    ]
    
    curr = Path.cwd().resolve()
    for _ in range(6):
        target = curr / 'AgroStats AndTech' / 'AgroStatsApp' / 'src'
        if target.exists() and (target / 'notebook_code').is_dir():
            candidates.insert(0, target)
            break
        curr = curr.parent

    for c in candidates:
        if c.exists() and (c / 'notebook_code').is_dir():
            resolved = str(c.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            print(f'[OK] Módulo src localizado localmente en: {resolved}')
            return

setup_environment()

# Importación del analizador exploratorio
from notebook_code import EdaAnalyzer, load_all_raw_datasets, RawDataLoader

print('[OK] Módulos de EDA importados exitosamente.')


In [ ]:
# 2. Carga de Datasets Crudos desde data/RAW
datasets = load_all_raw_datasets()
print(f'Total de datasets cargados: {len(datasets)}')
df_pluvio = datasets.get('ideam_pluvio')
df_precios = datasets.get('sipsa_precios')
print(f'• Pluviometría: {df_pluvio.shape if df_pluvio is not None else None}')
print(f'• Precios SIPSA: {df_precios.shape if df_precios is not None else None}')


In [ ]:
# 3. Estadísticos Distributivos Avanzados (Asimetría, Curtosis, IQR)
dist_stats = EdaAnalyzer.describe_numeric_distributions(df_pluvio)
display(dist_stats)


In [ ]:
# 4. Detección de Valores Atípicos y Anomalías (Criterio de Tukey)
outliers_report = EdaAnalyzer.detect_outliers(df_pluvio, method='tukey', threshold=1.5)
display(outliers_report)


In [ ]:
# 5. Análisis de Correlación y Colinealidad
corr_matrix, collin_pairs = EdaAnalyzer.correlation_analysis(df_pluvio, method='spearman')
print('=== TOP PARES DE CORRELACIÓN ===')
display(collin_pairs.head(10))


In [ ]:
# 6. Visualización: Mapa de Calor de Correlaciones
fig_corr = EdaAnalyzer.plot_correlation_heatmap(df_pluvio, title='Matriz de Correlación - Variables Pluviométricas')
if fig_corr:
    fig_corr.show()


In [ ]:
# 7. Dinámica Temporal de la Variable Principal
if 'fechaobservacion' in df_pluvio.columns and 'valorobservado' in df_pluvio.columns:
    fig_ts = EdaAnalyzer.plot_time_series(
        df_pluvio,
        date_col='fechaobservacion',
        value_col='valorobservado',
        title='Trayectoria Temporal de Precipitación Diaria (IDEAM)'
    )
    if fig_ts:
        fig_ts.show()
